[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/prompts/chain_of_thought_prompting.ipynb)


<center>
<p style="text-align:center">
<img alt="Phoenix 标志" src="https://raw.githubusercontent.com/Arize-ai/phoenix-assets/9e6101d95936f4bd4d390efc9ce646dc6937fb2d/images/socal/github-large-banner-phoenix.jpg" width="1000"/>
<br>
<br>
<a href="https://arize.com/docs/phoenix/">文档</a>
|
<a href="https://github.com/Arize-ai/phoenix">GitHub</a>
|
<a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
</p>
</center>
<h1 align="center">思维链提示教程</h1>

LLM 擅长文本生成，但推理能力会受到提示方式的影响。**思维链（CoT）**提示引导模型逐步思考，增强逻辑推理，从而提升数学、逻辑和多步骤问题求解任务的准确性。

本教程将：
- 在数据集上评估模型，观察不同提示技术对推理的影响。
- 从基础提示逐步改进为结构化推理策略。
- 使用 Phoenix 评估各阶段准确率，并探索模型的推理过程。
- 学习如何在实际任务中有效应用思维链提示。

⚠️ 本教程需要 OpenAI API 密钥。

开始吧！🚀

# **配置依赖与密钥**

> 添加引用块



In [ ]:
!pip install -qqqq arize-phoenix-client datasets openinference-instrumentation-openai

接下来连接 Phoenix。下面的代码连接到 Phoenix Cloud 实例；你也可以选择[连接自托管 Phoenix 实例](https://arize.com/docs/phoenix/deployment)。

In [ ]:
import os
from getpass import getpass

if not os.environ.get("PHOENIX_COLLECTOR_ENDPOINT"):
    os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = input(
        "请输入 Phoenix 采集端地址（例如 https://app.phoenix.arize.com/s/example）："
    )
if not os.environ.get("PHOENIX_API_KEY"):
    os.environ["PHOENIX_API_KEY"] = getpass("请输入 Phoenix API 密钥：")

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("请输入 OpenAI API 密钥：")

# **将数据集加载到 Phoenix**

此数据集包含数学应用题、逐步讲解及对应答案。改进提示词时，将使用它测试、衡量和跟踪效果提升。

这里还导入 Phoenix 客户端，以便直接在 Notebook 中创建和修改提示词，并将变化同步到 Phoenix 界面。

In [ ]:
import uuid

from datasets import load_dataset
from phoenix.client import Client as PhoenixClient

ds = load_dataset("syeddula/math_word_problems")["train"]
ds = ds.to_pandas()
ds.head()

unique_id = uuid.uuid4()

# 将数据集上传到 Phoenix
dataset = PhoenixClient().datasets.create_dataset(
    dataframe=ds,
    input_keys=["Word Problem"],
    output_keys=["Answer"],
    name=f"wordproblems-{unique_id}",
)

# **零样本提示**——基线

**零样本提示**是与语言模型交互的最简单方式：直接提问，不提供示例或推理步骤。模型仅根据预训练知识生成答案。

本节以此作为比较基线。通过数据集评估，可以观察模型在没有明确指导时解决数学应用题的能力。后续将引入**思维链（CoT）**等结构化推理技术，衡量准确率和答案的变化。

In [ ]:
from openai import OpenAI
from openai.types.chat.completion_create_params import CompletionCreateParamsBase
from phoenix.client.types import PromptVersion

params = CompletionCreateParamsBase(
    model="gpt-3.5-turbo",
    temperature=0,
    messages=[
        {
            "role": "system",
            "content": "你是数学应用题评估器。只输出整数答案，不要包含文字、解释、符号、标签或单位，小数答案一律四舍五入。",
        },
        {"role": "user", "content": "{{Problem}}"},
    ],
)

prompt_identifier = "wordproblems"

prompt = PhoenixClient().prompts.create(
    name=prompt_identifier,
    prompt_description="用于计算数学应用题答案的提示词。",
    version=PromptVersion.from_openai(params),
)

此时，初始提示词已出现在 Phoenix 的提示词页面。后续任何修改都会记录在**版本（Versions）**中，便于跟踪和比较变化。

Phoenix 存储的不只有文本，还包括提示词模板、模型配置和响应格式等关键信息，让输出生成过程保持结构化和一致性。

![界面中的基线提示词](https://storage.googleapis.com/arize-phoenix-assets/assets/images/CoT_Demo_1.png)

接下来为实验定义任务与评估器，再运行实验。

由于数据集含有真实标签，可以使用简单函数提取答案，判断计算结果是否符合预期。

In [ ]:
from phoenix.client import AsyncClient


def zero_shot_prompt(input):
    client = OpenAI()
    resp = client.chat.completions.create(
        **prompt.format(variables={"Problem": input["Word Problem"]})
    )
    return resp.choices[0].message.content.strip()


def evaluate_response(output, expected):
    if not output.isdigit():
        return False
    return int(output) == int(expected["Answer"])


async_client = AsyncClient()
initial_experiment = await async_client.experiments.run_experiment(
    dataset=dataset,
    task=zero_shot_prompt,
    evaluators=[evaluate_response],
    experiment_description="零样本提示",
    experiment_name="zero-shot-prompt",
    experiment_metadata={"prompt": "prompt_id=" + prompt.id},
)

可以在 Phoenix 中查看实验结果。本次运行约有 75% 的准确率。后续将迭代提示词，观察评测变化。

**注意：**本教程中的评测结果会有波动，有时可能低于初始实验。这不一定表示存在缺陷，模型选择、模型固有行为和随机性都可能导致结果变化。


![基线实验](https://storage.googleapis.com/arize-phoenix-assets/assets/images/CoT_Demo_2.png)

# **零样本思维链提示**

零样本提示会直接给出答案，但处理复杂推理时往往不够理想。**零样本思维链（CoT）**提示明确要求模型先逐步思考，再给出最终答案，以改善这一问题。

加入“让我们逐步思考”这样的简单指令，可鼓励模型按逻辑拆解问题。这种结构化推理可能提高准确性，尤其适用于多步数学题。

本节比较零样本思维链与基线，评估其影响。先创建提示词。

In [ ]:
zero_shot_COT_template = """
你是数学应用题评估器，负责输出题目的答案。

给出答案前，必须先按逻辑分析问题。

先展示部分推理过程。

然后在最后单独一行仅输出整数答案。最终答案不得包含文字、逗号、标签或单位，小数答案一律四舍五入。

"""

params = CompletionCreateParamsBase(
    model="gpt-3.5-turbo",
    temperature=0,
    messages=[
        {"role": "system", "content": zero_shot_COT_template},
        {"role": "user", "content": "{{Problem}}"},
    ],
)

zero_shot_COT = PhoenixClient().prompts.create(
    name=prompt_identifier,
    prompt_description="零样本思维链提示词",
    version=PromptVersion.from_openai(params),
)

更新后的提示词现已作为新版本保存在 Phoenix。

接下来运行任务和评测，从 LLM 输出中提取答案。

In [ ]:
import re


def zero_shot_COT_prompt(input):
    client = OpenAI()
    resp = client.chat.completions.create(
        **zero_shot_COT.format(variables={"Problem": input["Word Problem"]})
    )
    response_text = resp.choices[0].message.content.strip()
    lines = response_text.split("\n")
    final_answer = lines[-1].strip()
    final_answer = re.sub(r"^\*\*(\d+)\*\*$", r"\1", final_answer)
    return {"full_response": response_text, "final_answer": final_answer}


def evaluate_response(output, expected):
    final_answer = output["final_answer"]
    if not final_answer.isdigit():
        return False
    return int(final_answer) == int(expected["Answer"])


initial_experiment = await async_client.experiments.run_experiment(
    dataset=dataset,
    task=zero_shot_COT_prompt,
    evaluators=[evaluate_response],
    experiment_description="零样本思维链提示",
    experiment_name="zero-shot-cot-prompt",
    experiment_metadata={"prompt": "prompt_id=" + zero_shot_COT.id},
)

点击 Phoenix 中的实验，可以查看模型得到答案所经历的步骤。本次要求模型逐步思考并输出推理过程后，效果有所提升。

![实验详情](https://storage.googleapis.com/arize-phoenix-assets/assets/images/CoT_Demo_3.png)

![第二次实验结果](https://storage.googleapis.com/arize-phoenix-assets/assets/images/CoT_Demo_4.png)

# **自洽性思维链提示**

即使用了思维链提示，单次回答也未必可靠。**自洽性思维链**通过生成多条推理路径并选择最常见的答案来提高准确性。它对多个输出进行采样与聚合，而非依赖单次回答，以减少随机性或推理错误造成的影响。

这种方式有助于提高稳健性，尤其适合初始推理步骤可能不同的复杂问题。本节将其与前面的提示词比较，观察多个回答对总体效果的影响。

使用新提示词重复前面的流程并评估结果。

In [ ]:
consistency_COT_template = """

你是数学应用题评估器，负责输出题目的答案。

请按以下步骤操作：
1. **多次独立解题**，每次都仔细分析求解过程。
2. 为每次独立尝试展示部分推理过程。
3. 找出多次尝试中出现次数最多的整数答案。
4. 在**新的一行**仅输出该多数答案，使用纯整数，**不含文字、逗号、标签、单位或特殊字符**。
"""

params = CompletionCreateParamsBase(
    model="gpt-3.5-turbo",
    temperature=0,
    messages=[
        {"role": "system", "content": consistency_COT_template},
        {"role": "user", "content": "{{Problem}}"},
    ],
)

self_consistency_COT = PhoenixClient().prompts.create(
    name=prompt_identifier,
    prompt_description="自洽性思维链提示词",
    version=PromptVersion.from_openai(params),
)

In [ ]:
def self_consistency_COT_prompt(input):
    client = OpenAI()
    resp = client.chat.completions.create(
        **self_consistency_COT.format(variables={"Problem": input["Word Problem"]})
    )
    response_text = resp.choices[0].message.content.strip()
    lines = response_text.split("\n")
    final_answer = lines[-1].strip()
    final_answer = re.sub(r"^\*\*(\d+)\*\*$", r"\1", final_answer)
    return {"full_response": response_text, "final_answer": final_answer}


def evaluate_response(output, expected):
    final_answer = output["final_answer"]
    if not final_answer.isdigit():
        return False
    return int(final_answer) == int(expected["Answer"])


initial_experiment = await async_client.experiments.run_experiment(
    dataset=dataset,
    task=self_consistency_COT_prompt,
    evaluators=[evaluate_response],
    experiment_description="自洽性思维链提示",
    experiment_name="self-consistency-cot-prompt",
    experiment_metadata={"prompt": "prompt_id=" + self_consistency_COT.id},
)

本次观察到明显提升！由于提示词要求模型独立计算多次，实验运行时间可能稍长。点击实验浏览页面，可查看模型针对每道题进行的独立计算。


![第二轮结果](https://storage.googleapis.com/arize-phoenix-assets/assets/images/CoT_Demo_6.png)

![第二轮实验](https://storage.googleapis.com/arize-phoenix-assets/assets/images/CoT_Demo_5.png)

# 少样本思维链提示

**少样本思维链提示**先提供完整解题示例，再让模型解决新问题，以增强推理能力。逐步解题示范可帮助模型在未见过的问题上运用类似逻辑。

这种方式利用**上下文学习**，让模型从示例中归纳规律。

最后比较少样本思维链与前面的提示词。

先从测试数据集中抽取示例，构建提示词。

In [ ]:
ds = load_dataset("syeddula/math_word_problems")["test"]
few_shot_examples = ds.to_pandas().sample(5)
few_shot_examples

现在构建最终提示词、运行实验并查看结果。可以在 Phoenix 的**提示词页面**跟踪版本历史，并查看随机选取的示例。

In [ ]:
few_shot_COT_template = """
你是数学应用题评估器，负责输出题目的答案。 给出答案前，必须先按逻辑分析问题。 展示部分推理过程。

最后在单独一行仅输出整数答案。最终答案不得包含文字、逗号、标签或单位，小数答案一律四舍五入。

以下数学应用题、逐步解释和答案示例供推理参考：

{examples}
"""
params = CompletionCreateParamsBase(
    model="gpt-3.5-turbo",
    temperature=0,
    messages=[
        {"role": "system", "content": few_shot_COT_template.format(examples=few_shot_examples)},
        {"role": "user", "content": "{{Problem}}"},
    ],
)

few_shot_COT = PhoenixClient().prompts.create(
    name=prompt_identifier,
    prompt_description="少样本思维链提示词",
    version=PromptVersion.from_openai(params),
)

![少样本提示词](https://storage.googleapis.com/arize-phoenix-assets/assets/images/CoT_Demo_8.png)

In [ ]:
def few_shot_COT_prompt(input):
    client = OpenAI()
    resp = client.chat.completions.create(
        **few_shot_COT.format(variables={"Problem": input["Word Problem"]})
    )
    response_text = resp.choices[0].message.content.strip()
    lines = response_text.split("\n")
    final_answer = lines[-1].strip()
    final_answer = re.sub(r"^\*\*(\d+)\*\*$", r"\1", final_answer)
    return {"full_response": response_text, "final_answer": final_answer}


def evaluate_response(output, expected):
    final_answer = output["final_answer"]
    if not final_answer.isdigit():
        return False
    return int(final_answer) == int(expected["Answer"])


initial_experiment = await async_client.experiments.run_experiment(
    dataset=dataset,
    task=few_shot_COT_prompt,
    evaluators=[evaluate_response],
    experiment_description="少样本思维链提示",
    experiment_name="few-shot-cot-prompt",
    experiment_metadata={"prompt": "prompt_id=" + few_shot_COT.id},
)

# **最终结果**

完成全部实验后，可以比较不同提示技术的表现。请注意，随机性和模型非确定性行为可能导致结果波动。

在**提示词页面**查看版本历史，也可以进入**试验场（Playground）**继续迭代并运行更多实验。

要进一步改进并在其他数据集上测试，可探索思维链与具体场景的适配程度。Phoenix 的 TypeScript 和 Python 客户端可将此过程接入工作流。


![最终运行结果](https://storage.googleapis.com/arize-phoenix-assets/assets/images/CoT_Demo_7.png)

接下来可以查看更多 [Phoenix 示例](https://arize.com/docs/phoenix/notebooks)。如果还没有为项目加星，也欢迎[在 GitHub 上支持我们](https://github.com/Arize-ai/phoenix)！⭐️